# 🟢 Lab 03 — Green badge, missing episode
A fake stream of viewing events has a contract of 100 events per batch. This job writes 100 rows with `mode=baseline` and only 10 with `mode=thin`. Both return success. Use a new `batch_id` for each batch, and reuse an ID only to replay the same batch. The notebook reports the shortfall without failing on purpose. In production this is where a hard data contract could stop publication.

Try `mode=baseline, batch_id=baseline_01`, then `mode=thin, batch_id=thin_01` for an immediate SQL demonstration. If doing a real anomaly trial, build a baseline before your first thin run using the protocol below.


In [ ]:
%sql
SELECT assert_true(:catalog RLIKE '^[A-Za-z_][A-Za-z0-9_]*$' AND :schema RLIKE '^zeroops_lab_[a-z0-9_]+$', 'Use a dedicated zeroops_lab_<alias> schema'),
       assert_true(:mode IN ('baseline', 'thin'), 'Use baseline or thin'),
       assert_true(:batch_id RLIKE '^[a-z0-9][a-z0-9_-]{0,63}$', 'Use a simple unique batch ID');


In [ ]:
%sql
CREATE TABLE IF NOT EXISTS IDENTIFIER(:catalog || '.' || :schema || '.zops_viewing_events') (
  batch_id STRING NOT NULL, event_id BIGINT NOT NULL, batch_mode STRING NOT NULL, ingested_at TIMESTAMP NOT NULL
) USING DELTA COMMENT 'Synthetic lab events. Grain: batch_id and event_id. Contract: 100 rows per batch.';


In [ ]:
%sql
SELECT assert_true(COUNT_IF(batch_mode <> :mode) = 0, 'Use a new batch_id when changing mode') AS replay_check
FROM IDENTIFIER(:catalog || '.' || :schema || '.zops_viewing_events')
WHERE batch_id = :batch_id;


In [ ]:
%sql
MERGE INTO IDENTIFIER(:catalog || '.' || :schema || '.zops_viewing_events') AS target
USING (SELECT :batch_id AS batch_id, id AS event_id, :mode AS batch_mode, current_timestamp() AS ingested_at
       FROM range(100) WHERE id < CASE WHEN :mode = 'thin' THEN 10 ELSE 100 END) AS batch
ON target.batch_id = batch.batch_id AND target.event_id = batch.event_id
WHEN NOT MATCHED THEN INSERT (batch_id, event_id, batch_mode, ingested_at)
VALUES (batch.batch_id, batch.event_id, batch.batch_mode, batch.ingested_at);


In [ ]:
%sql
SELECT :batch_id AS batch_id, :mode AS batch_mode, 100 AS expected_rows, COUNT(*) AS observed_rows,
       MIN(ingested_at) AS first_ingested_at, COUNT(DISTINCT event_id) AS distinct_rows,
       CASE WHEN COUNT(*) = 100 AND COUNT(DISTINCT event_id) = 100 THEN 'PASS' ELSE 'UNDERFILLED_OR_DUPLICATE' END AS contract_status
FROM IDENTIFIER(:catalog || '.' || :schema || '.zops_viewing_events')
WHERE batch_id = :batch_id;


In [ ]:
%sql
DESCRIBE HISTORY IDENTIFIER(:catalog || '.' || :schema || '.zops_viewing_events') LIMIT 10;


## Optional — Ask the anomaly detector to earn its popcorn
Enable separately billed Data Quality Monitoring anomaly detection on the disposable schema, and include the table in the ZeroOps scope. Use unique `baseline` batches on a real, regular cadence across multiple days. Wait for the Quality tab to finish Training. Then substitute thin batches for the usual writes over an observation window so a healthy batch in the same 24 hours cannot mask the drop. Check the Quality tab and Delta history before looking for a ZeroOps insight.

Optional freshness variant: stop the writes after a regular baseline and observe the next scan. A fresh commit can carry old event timestamps. This experiment measures commit freshness. Do not modify event timestamps to pretend time has elapsed. Log separately the bad commit time, the scan, and any insight. If the table is still training, record pending. A fast two-run demonstration only proves the SQL control and the misleading green badge.

[Databricks anomaly detection](https://docs.databricks.com/aws/en/data-governance/unity-catalog/data-quality-monitoring/anomaly-detection)
